# Lab 4.3 — NLP Ticket & Log Classifier
**Module 4: Applied Natural Language Processing — Hands-on Lab**

This lab builds the complete end-to-end NLP pipeline covering both production use cases:

- **Use Case A — Ticket Routing:** Classify incoming Nutanix support tickets by category and route them to the correct team queue
- **Use Case B — Log Classification:** Parse AOS/Prism error log strings and classify them by error family for automated alerting
- **Combined Pipeline:** A single `NutanixNLPPipeline` class that handles both inputs and exports as a joblib artefact

> **Instructor Note:** This is where everything from Labs 4.1 and 4.2 comes together. The ticket classifier maps to a real Nutanix L1/L2 workflow — auto-routing saves ~15 minutes per ticket. The log classifier is a building block for AIOps: instead of a Prism alert, the system can now read a raw AOS log line and know it is an IO_ERROR that should page the storage team.

## 📦 Requirements & Troubleshooting

### Required Packages

| Package | Install Name | Notes |
|---------|-------------|-------|
| nltk | `nltk` | Also downloads `punkt`, `stopwords`, `wordnet` data |
| scikit-learn | `scikit-learn` | |
| pandas | `pandas` | |
| numpy | `numpy` | |
| joblib | `joblib` | |
| matplotlib | `matplotlib` | |

**Install all at once:**
```bash
pip install nltk scikit-learn pandas numpy joblib matplotlib
```

---

### ⚠️ Common Errors & Fixes

**`ModuleNotFoundError: No module named '...'`**
> Package is missing from the active Python environment.
> Fix: Run the pip install command above in a terminal, then **restart the kernel**.

**`CalledProcessError` — `--break-system-packages` / exit status 2**
> You are using a virtual environment (e.g. `myenv`) where that flag is not supported, or your pip version is old.
> Fix: Open a terminal, activate your venv (`source myenv/bin/activate`), then run `pip install <package>` without that flag.

**`Failed building wheel for <package>` / C extension errors**
> The package does not support your Python version (most common on Python 3.14).
> Fix: Switch the kernel to **Python 3.13**. Click the kernel name in the VS Code top-right corner → *Select Another Kernel* → *Python 3.13*. Then re-run.

**Packages install with no error but `ModuleNotFoundError` still appears**
> You installed into a different Python than the one the notebook is using.
> Fix: Check the kernel shown in the top-right of VS Code. Open a terminal, activate that environment, and install packages there.

**`PermissionError` or `[Errno 13]` when installing**
> Trying to install into a read-only system Python.
> Fix: Use a virtual environment — `python -m venv myenv && source myenv/bin/activate` — then install.


In [2]:
import subprocess, sys

required = {
    'nltk': 'nltk',
    'sklearn': 'scikit-learn',
    'pandas': 'pandas',
    'numpy': 'numpy',
    'joblib': 'joblib',
    'matplotlib': 'matplotlib',
}
for pkg, inst in required.items():
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', inst, '--quiet', '--break-system-packages'])

import nltk, warnings
for r in ['punkt', 'punkt_tab', 'stopwords', 'wordnet']:
    nltk.download(r, quiet=True)
warnings.filterwarnings('ignore')
print('All packages ready ✅')

All packages ready ✅


## Use Case A — IT Support Ticket Classification & Routing

### A.1 Dataset

We reuse the 60-ticket dataset from Lab 4.1/4.2 and extend it with a **routing map** — the team queue each category should be sent to.

> **Instructor Note:** In a real deployment this routing map would be in a database and editable by operations managers. The NLP model only outputs the category; the routing rule is separate business logic. This separation is good design — you can change routing rules without retraining the model.

In [3]:
import os, re, json
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib

# ── Routing table: category → team queue and SLA tier ──────────────────────
ROUTING_TABLE = {
    'storage':     {'queue': 'STOR-L2',    'team': 'Storage Engineering',     'sla_hours': 4},
    'network':     {'queue': 'NET-L2',     'team': 'Network Operations',       'sla_hours': 2},
    'compute':     {'queue': 'CMP-L2',     'team': 'Compute & Hypervisor',     'sla_hours': 4},
    'prism':       {'queue': 'PRISM-L1',   'team': 'Prism UI & API Support',   'sla_hours': 8},
    'replication': {'queue': 'DR-L2',      'team': 'Data Protection & DR',     'sla_hours': 2},
    'performance': {'queue': 'PERF-L2',    'team': 'Performance Engineering',  'sla_hours': 8},
}

print('Routing table:')
for cat, info in ROUTING_TABLE.items():
    print(f'  {cat:<14} → queue={info["queue"]:<12} team={info["team"]}  SLA={info["sla_hours"]}h')

Routing table:
  storage        → queue=STOR-L2      team=Storage Engineering  SLA=4h
  network        → queue=NET-L2       team=Network Operations  SLA=2h
  compute        → queue=CMP-L2       team=Compute & Hypervisor  SLA=4h
  prism          → queue=PRISM-L1     team=Prism UI & API Support  SLA=8h
  replication    → queue=DR-L2        team=Data Protection & DR  SLA=2h
  performance    → queue=PERF-L2      team=Performance Engineering  SLA=8h


### A.2 Preprocessing Pipeline

In [4]:
ALL_STOPS = set(stopwords.words('english')) | {
    'nutanix', 'node', 'cluster', 'vm', 'vms', 'issue', 'error',
    'please', 'hi', 'hello', 'ticket', 'problem', 'thank', 'thanks'
}
_lem = WordNetLemmatizer()

def preprocess(text: str, min_len: int = 3) -> str:
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if re.match(r'^[a-z]+$', t)]
    tokens = [t for t in tokens if t not in ALL_STOPS]
    tokens = [_lem.lemmatize(t, pos='v') for t in tokens]
    return ' '.join([t for t in tokens if len(t) >= min_len])

print('Preprocessing function ready.')
print(f'Example: "Stargate OOM killed after memory pressure" → "{preprocess("Stargate OOM killed after memory pressure")}"')

Preprocessing function ready.
Example: "Stargate OOM killed after memory pressure" → "stargate oom kill memory pressure"


### A.3 Train the Ticket Classifier

In [5]:
# Load from Lab 4.1 CSV if available; else rebuild inline
CSV_PATH = 'tickets_preprocessed.csv'

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print(f'✅ Loaded {len(df)} preprocessed tickets from {CSV_PATH}')
else:
    raw_tickets = [
        {'text': 'Stargate process crashed on CVM node-3 after disk sda3 threw multiple IO errors during peak hours', 'category': 'storage'},
        {'text': 'NFS datastore becomes inaccessible intermittently, getting EXT4-fs errors in kernel log', 'category': 'storage'},
        {'text': 'Disk tier migration from SSD to HDD is stuck at 45% for more than 6 hours', 'category': 'storage'},
        {'text': 'Data path throughput dropped by 60% after replacing failed SSD on node-2', 'category': 'storage'},
        {'text': 'Storage pool utilization reached 92%, curator scan failing to reclaim space', 'category': 'storage'},
        {'text': 'Stargate OOM killed after memory pressure during large sequential writes to HDD tier', 'category': 'storage'},
        {'text': 'NDFS mount failed on AHV host, unable to access any VMs on that datastore', 'category': 'storage'},
        {'text': 'Disk bad sectors detected on SATA drive in node-1, Stargate marked drive as unhealthy', 'category': 'storage'},
        {'text': 'Data locality policy not being enforced after cluster expansion with two new nodes added', 'category': 'storage'},
        {'text': 'Read performance degraded after rolling upgrade to AOS 6.5, IOPS dropped significantly', 'category': 'storage'},
        {'text': '10GbE NIC on node-4 shows frequent link flaps causing packet loss for running VMs', 'category': 'network'},
        {'text': 'AHV host cannot reach default gateway after OVS bridge configuration change', 'category': 'network'},
        {'text': 'LACP bond interface went down on node-2 after upstream switch firmware update', 'category': 'network'},
        {'text': 'VM network connectivity intermittently lost, OVS flow table shows duplicate MAC entries', 'category': 'network'},
        {'text': 'BGP session dropping between cluster and ToR switch causing VIP failover events', 'category': 'network'},
        {'text': 'Jumbo frames not accepted on storage network, MTU mismatch causing Stargate packet drops', 'category': 'network'},
        {'text': 'Ping latency from CVM to AHV host spiking to 50ms, expected sub-millisecond on local interface', 'category': 'network'},
        {'text': 'IPAM address pool exhausted on VM network, new VM deployments failing with IP allocation error', 'category': 'network'},
        {'text': 'Flow network policy blocking legitimate VM to VM traffic on the same subnet', 'category': 'network'},
        {'text': 'CVM external IP unreachable from management network after VLAN reconfiguration last night', 'category': 'network'},
        {'text': 'VM fails to power on, not enough memory available on AHV host after adding four new VMs', 'category': 'compute'},
        {'text': 'AHV live migration fails with dirty page transfer timeout after 300 seconds elapsed', 'category': 'compute'},
        {'text': 'CVM memory usage at 98%, acropolis service consuming unexpectedly high heap memory', 'category': 'compute'},
        {'text': 'Guest VM showing blue screen after vCPU hot-add operation on Windows Server 2019', 'category': 'compute'},
        {'text': 'CPU ready time spiking above 20% on overcommitted host, virtual machine performance degraded', 'category': 'compute'},
        {'text': 'AHV host entered maintenance mode unexpectedly during workload, VMs migrated to other nodes', 'category': 'compute'},
        {'text': 'VM clone operation failing at 30% with qcow2 image corruption error in acropolis log', 'category': 'compute'},
        {'text': 'NUMA topology not respected for latency sensitive database VM after node restart', 'category': 'compute'},
        {'text': 'Balloon driver inflating inside guest causing application out of memory inside Windows VM', 'category': 'compute'},
        {'text': 'VirtIO network driver crash inside guest causing virtual machine to become unresponsive', 'category': 'compute'},
        {'text': 'Prism Central UI shows blank page after login, browser console shows 502 bad gateway error', 'category': 'prism'},
        {'text': 'REST API v3 /vms endpoint returning 504 timeout for requests listing more than 100 VMs', 'category': 'prism'},
        {'text': 'Prism Element cluster health shows warning state but no specific alert details are visible', 'category': 'prism'},
        {'text': 'Alert email notifications stopped working after SMTP server IP address change in Prism settings', 'category': 'prism'},
        {'text': 'Prism Central data collection job is stuck, analytics graphs show stale data from 3 hours ago', 'category': 'prism'},
        {'text': 'SSO login redirecting to wrong Prism Central instance after DNS change in the environment', 'category': 'prism'},
        {'text': 'REST API rate limiting triggered at 100 requests per minute affecting automation scripts', 'category': 'prism'},
        {'text': 'Prism Central upgrade from 2022 to 2023 failed at post upgrade health check step', 'category': 'prism'},
        {'text': 'Category management API returning 403 forbidden for non-admin user despite correct RBAC policy', 'category': 'prism'},
        {'text': 'Capacity planning chart showing incorrect storage runway calculation for SSD tier', 'category': 'prism'},
        {'text': 'Cerebro async replication failing between primary and recovery sites, lag exceeded 4 hours', 'category': 'replication'},
        {'text': 'Synchronous replication partner site unreachable, protection domain status showing error state', 'category': 'replication'},
        {'text': 'Snapshot replication schedule not triggering at configured 15 minute interval', 'category': 'replication'},
        {'text': 'Metro availability failover test failed, witness VM not responding to heartbeat probe', 'category': 'replication'},
        {'text': 'DR runbook automation failed to power on virtual machines in recovery site after planned failover', 'category': 'replication'},
        {'text': 'Cerebro replication bandwidth throttled to 10Mbps, expected 1Gbps during batch replication window', 'category': 'replication'},
        {'text': 'Consistency group snapshot incomplete, some VMs excluded because quiescing operation timed out', 'category': 'replication'},
        {'text': 'Replication job consuming 100% of WAN link causing production latency increase during business hours', 'category': 'replication'},
        {'text': 'Protection domain recovery point objective showing 6 hours instead of configured 1 hour RPO target', 'category': 'replication'},
        {'text': 'NearSync replication health degraded after adding three new VMs to the protection domain yesterday', 'category': 'replication'},
        {'text': 'Storage latency spiking to 40 milliseconds for random 4K write operations during business hours', 'category': 'performance'},
        {'text': 'SQL Server workload IOPS capped at 5000, cluster should deliver 80000 IOPS per node', 'category': 'performance'},
        {'text': 'CVM IO scheduler priority configuration not applied correctly after cluster upgrade last week', 'category': 'performance'},
        {'text': 'Erasure coding rebuild consuming excessive disk IO causing noisy neighbour issue for production VMs', 'category': 'performance'},
        {'text': 'SSD cache hit ratio dropped from 85 to 30 percent after adding new unoptimised sequential workload', 'category': 'performance'},
        {'text': 'Network throughput bottleneck on 10GbE storage network during nightly backup window', 'category': 'performance'},
        {'text': 'Compression ratio degraded after enabling deduplication, overall capacity savings significantly reduced', 'category': 'performance'},
        {'text': 'AOS upgrade caused 30 percent throughput regression for all VMs lasting 2 hours post upgrade', 'category': 'performance'},
        {'text': 'Garbage collection in Stargate causing periodic write latency spikes every 4 minutes', 'category': 'performance'},
        {'text': 'Memory tiering performance poor after enabling hardware memory encryption on AMD EPYC nodes', 'category': 'performance'},
    ]
    df = pd.DataFrame(raw_tickets)
    df['clean_text'] = df['text'].apply(preprocess)
    print(f'⚠️  Rebuilt inline ({len(df)} tickets). Run Lab 4.1 first for the CSV.')

X_text = df['clean_text'].values
y_cat  = df['category'].values

X_train, X_test, y_train, y_test = train_test_split(
    X_text, y_cat, test_size=0.25, random_state=42, stratify=y_cat
)

# TF-IDF + LinearSVC (best from Lab 4.2)
ticket_vec = TfidfVectorizer(max_features=200, ngram_range=(1, 2), sublinear_tf=True)
ticket_clf = CalibratedClassifierCV(LinearSVC(C=1.0, max_iter=2000, random_state=42), cv=3)

X_train_vec = ticket_vec.fit_transform(X_train)
X_test_vec  = ticket_vec.transform(X_test)

ticket_clf.fit(X_train_vec, y_train)
preds = ticket_clf.predict(X_test_vec)
acc   = accuracy_score(y_test, preds)

print(f'\nTicket classifier accuracy: {acc:.2%}')
print(classification_report(y_test, preds))

✅ Loaded 60 preprocessed tickets from tickets_preprocessed.csv

Ticket classifier accuracy: 33.33%
              precision    recall  f1-score   support

     compute       0.25      0.50      0.33         2
     network       0.00      0.00      0.00         2
 performance       0.25      0.33      0.29         3
       prism       0.67      0.67      0.67         3
 replication       1.00      0.33      0.50         3
     storage       0.00      0.00      0.00         2

    accuracy                           0.33        15
   macro avg       0.36      0.31      0.30        15
weighted avg       0.42      0.33      0.33        15



### A.4 Ticket Routing Function

> **Instructor Note:** This is the production-facing interface. It takes a raw ticket string (exactly as a user would submit it), runs the full preprocessing + classification pipeline, and returns a structured routing decision. The confidence threshold (0.4) prevents the model from confidently routing an ambiguous ticket — it falls back to a manual triage queue instead.

In [6]:
def route_ticket(raw_text: str, confidence_threshold: float = 0.4) -> dict:
    """Classify a raw support ticket and return a routing decision."""
    clean = preprocess(raw_text)
    vec   = ticket_vec.transform([clean])

    category   = ticket_clf.predict(vec)[0]
    proba      = ticket_clf.predict_proba(vec)[0]
    confidence = float(proba.max())

    if confidence < confidence_threshold:
        return {
            'category':   'unknown',
            'queue':      'L1-TRIAGE',
            'team':       'L1 Manual Triage',
            'sla_hours':  8,
            'confidence': round(confidence, 3),
            'action':     'MANUAL_REVIEW',
        }

    routing = ROUTING_TABLE[category]
    return {
        'category':   category,
        'queue':      routing['queue'],
        'team':       routing['team'],
        'sla_hours':  routing['sla_hours'],
        'confidence': round(confidence, 3),
        'action':     'AUTO_ROUTE',
    }


# Test routing on sample tickets
test_tickets = [
    "Stargate has crashed on node-2, we are seeing IOPS drop to zero for all VMs",
    "Cannot access Prism Central, getting 502 error, already tried restarting the PC VM",
    "Cerebro replication lag is 8 hours and growing, RPO breach is imminent",
    "NIC flapping on node-3 causing packet drops on the 10GbE storage interface",
    "AHV live migration stuck, VM has been migrating for 45 minutes with no progress",
    "Write latency suddenly jumped from 1ms to 35ms after last nights AOS upgrade",
    "Something is broken",  # ambiguous — should go to manual triage
]

print(f'{"Ticket (truncated)":<52} {"Category":<13} {"Queue":<12} {"Conf":<6} {"Action"}')
print('-' * 105)
for ticket in test_tickets:
    result = route_ticket(ticket)
    short  = ticket[:50] + '...' if len(ticket) > 50 else ticket
    print(f'{short:<52} {result["category"]:<13} {result["queue"]:<12} {result["confidence"]:<6} {result["action"]}')

Ticket (truncated)                                   Category      Queue        Conf   Action
---------------------------------------------------------------------------------------------------------
Stargate has crashed on node-2, we are seeing IOPS... storage       STOR-L2      0.476  AUTO_ROUTE
Cannot access Prism Central, getting 502 error, al... prism         PRISM-L1     0.784  AUTO_ROUTE
Cerebro replication lag is 8 hours and growing, RP... replication   DR-L2        0.696  AUTO_ROUTE
NIC flapping on node-3 causing packet drops on the... network       NET-L2       0.703  AUTO_ROUTE
AHV live migration stuck, VM has been migrating fo... unknown       L1-TRIAGE    0.339  MANUAL_REVIEW
Write latency suddenly jumped from 1ms to 35ms aft... performance   PERF-L2      0.588  AUTO_ROUTE
Something is broken                                  unknown       L1-TRIAGE    0.218  MANUAL_REVIEW


## Use Case B — AOS/Prism Error Log Classification by Error Family

### B.1 AOS Error Log Dataset

AOS logs follow a structured but verbose format. Each line contains a severity level, service name, file reference, and a free-text message. We classify these into six error families that map to different on-call escalation paths.

| Error Family | Trigger | On-call team |
|-------------|---------|-------------|
| `IO_ERROR` | Disk I/O failures, Stargate WAL corruption | Storage on-call |
| `REPLICATION_ERROR` | Cerebro lag, snapshot failures, witness timeout | DR on-call |
| `NETWORK_ERROR` | NIC flap, OVS drops, MTU mismatch | Network on-call |
| `MEMORY_ERROR` | OOM kill, balloon inflation, heap dump | Platform on-call |
| `HARDWARE_ERROR` | Fan failure, PSU failure, ECC errors | Hardware on-call |
| `AUTH_ERROR` | SSO failure, cert expiry, RBAC deny | Security on-call |

> **Instructor Note:** Notice that log classification uses the same TF-IDF + SVM pipeline as ticket classification — the representation is identical. The difference is that log strings have structured prefixes (severity, service) that we *do not* discard. These are actually strong features — `FATAL stargate` almost always means IO_ERROR.

In [7]:
aos_logs = [
    # IO_ERROR
    {'text': 'FATAL stargate disk_manager disk_id sda3 io_error sectors retry_count marking_disk_bad', 'family': 'IO_ERROR'},
    {'text': 'ERROR stargate data_path_io_error disk dev_sdb err EIO op WRITE offset timeout_exceeded', 'family': 'IO_ERROR'},
    {'text': 'FATAL disk node_id nvme smart_reallocated_sectors threshold marking_disk_bad unhealthy', 'family': 'IO_ERROR'},
    {'text': 'ERROR vdisk_controller vdisk_id io_error during_read latency_ms timeout disk_stall', 'family': 'IO_ERROR'},
    {'text': 'WARN stargate disk_WAL corruption detected offset checksum_mismatch data_integrity_risk', 'family': 'IO_ERROR'},
    {'text': 'FATAL stargate disk_write_error SATA drive bad_sector_count exceeded threshold IO_failure', 'family': 'IO_ERROR'},
    {'text': 'ERROR stargate extent_store read_timeout disk_latency_ms high retry_exceeded giving_up', 'family': 'IO_ERROR'},
    {'text': 'WARN disk_monitor SMART attribute reallocated_sectors_count critical_threshold approaching', 'family': 'IO_ERROR'},
    # REPLICATION_ERROR
    {'text': 'ERROR cerebro replication_manager site DR lag_seconds threshold exceeded protection_domain', 'family': 'REPLICATION_ERROR'},
    {'text': 'FATAL cerebro snapshot_sender snapshot_id dest_site remote_dr transfer_failed network_timeout', 'family': 'REPLICATION_ERROR'},
    {'text': 'ERROR cerebro consistency_group vms quiesce_timeout snapshot_skipped RPO_breach_risk', 'family': 'REPLICATION_ERROR'},
    {'text': 'WARN cerebro bandwidth_throttle site current_bw_mbps configured_bw_mbps throttle_active', 'family': 'REPLICATION_ERROR'},
    {'text': 'FATAL cerebro witness_heartbeat witness_ip last_seen_secs metro_cluster_at_risk failover', 'family': 'REPLICATION_ERROR'},
    {'text': 'ERROR cerebro NearSync replication_lag degraded protection_domain VMs_added sync_broken', 'family': 'REPLICATION_ERROR'},
    {'text': 'FATAL cerebro async_replication site_unreachable failover_pending manual_intervention_required', 'family': 'REPLICATION_ERROR'},
    {'text': 'WARN cerebro snapshot_schedule missed_window next_attempt retry_count RPO_target_risk', 'family': 'REPLICATION_ERROR'},
    # NETWORK_ERROR
    {'text': 'ERROR ovs_vswitchd node port link_state DOWN reason physical_carrier_lost nic_flap_count', 'family': 'NETWORK_ERROR'},
    {'text': 'FATAL ahv_networking bond lacp_pdu_timeout partner_key interface fallback_mode_active degraded', 'family': 'NETWORK_ERROR'},
    {'text': 'ERROR stargate network_io peer_cvm connection_refused retry storage_network_degraded packet_loss', 'family': 'NETWORK_ERROR'},
    {'text': 'WARN ovs flow_table duplicate_entry src_mac vlan dropping_traffic loop_risk spanning_tree', 'family': 'NETWORK_ERROR'},
    {'text': 'FATAL cvm eth mtu_mismatch local_mtu storage_vlan_mtu jumbo_frames_disabled switch_config', 'family': 'NETWORK_ERROR'},
    {'text': 'ERROR network bgp_session_down ToR_switch VIP_failover cluster_unreachable routing_table_invalid', 'family': 'NETWORK_ERROR'},
    {'text': 'WARN ovs_vswitchd packet_drops port rx_dropped tx_dropped buffer_overflow NIC_saturation', 'family': 'NETWORK_ERROR'},
    {'text': 'FATAL cvm external_ip unreachable management_network VLAN_mismatch gateway_unreachable', 'family': 'NETWORK_ERROR'},
    # MEMORY_ERROR
    {'text': 'FATAL kernel OOM_killer invoked process stargate pid rss available_mem oom_score killed', 'family': 'MEMORY_ERROR'},
    {'text': 'ERROR acropolis memory_pressure host balloon_inflation_mb guest_oom_events vm_paused', 'family': 'MEMORY_ERROR'},
    {'text': 'WARN cvm memory_usage swap_used_gb process chronos_master top_consumer oom_risk critical', 'family': 'MEMORY_ERROR'},
    {'text': 'FATAL kernel huge_page_allocation failed size order zone Normal fragmentation_high swap_storm', 'family': 'MEMORY_ERROR'},
    {'text': 'ERROR genesis memory_tracker heap_allocated_gb threshold triggering_heap_dump OOM_imminent', 'family': 'MEMORY_ERROR'},
    {'text': 'WARN cvm acropolis high_memory_usage balloon_driver inflating guest application_oom_inside', 'family': 'MEMORY_ERROR'},
    # HARDWARE_ERROR
    {'text': 'FATAL ipmi sensor node sensor FAN reading RPM threshold hardware_failure_detected chassis', 'family': 'HARDWARE_ERROR'},
    {'text': 'ERROR hades disk_diagnosis disk smart_test FAIL reallocated_sector_count remove_disk RMA', 'family': 'HARDWARE_ERROR'},
    {'text': 'FATAL chassis PSU power_supply_failure node redundancy_lost single_PSU_mode_active critical', 'family': 'HARDWARE_ERROR'},
    {'text': 'WARN hardware_monitor DIMM correctable_ecc_errors threshold degraded_performance memory_RMA', 'family': 'HARDWARE_ERROR'},
    {'text': 'FATAL bmc node_unreachable power_cycle_required chassis_intrusion_detected physical_access', 'family': 'HARDWARE_ERROR'},
    {'text': 'ERROR hardware NIC_hardware_failure port link_down physical_layer transceiver_malfunction RMA', 'family': 'HARDWARE_ERROR'},
    # AUTH_ERROR
    {'text': 'ERROR prism_gateway SSO redirect_uri_mismatch expected got login_failed certificate_mismatch', 'family': 'AUTH_ERROR'},
    {'text': 'FATAL certificate validation ssl_handshake_failed prism_central cert_expired days_overdue', 'family': 'AUTH_ERROR'},
    {'text': 'ERROR api_gateway user role resource method forbidden rbac_deny policy_violation audit_log', 'family': 'AUTH_ERROR'},
    {'text': 'WARN ldap_connector dc bind_failed timeout check_ldap_server_connectivity directory_unreachable', 'family': 'AUTH_ERROR'},
    {'text': 'FATAL prism_element local_auth lockout user admin failed_attempts lockout_duration account_locked', 'family': 'AUTH_ERROR'},
    {'text': 'ERROR prism SSO_login_redirecting wrong_instance DNS_change environment misconfiguration', 'family': 'AUTH_ERROR'},
]

log_df = pd.DataFrame(aos_logs)
print(f'Log dataset: {len(log_df)} entries, {log_df["family"].nunique()} error families')
print(log_df['family'].value_counts().to_string())

Log dataset: 42 entries, 6 error families
family
IO_ERROR             8
REPLICATION_ERROR    8
NETWORK_ERROR        8
MEMORY_ERROR         6
HARDWARE_ERROR       6
AUTH_ERROR           6


### B.2 Regex Pre-classifier

Before invoking the ML model, we apply a fast regex pre-classifier. AOS log lines often contain highly discriminative keywords (`OOM_killer`, `cerebro`, `ssl_handshake_failed`) that are near-certain signals for a specific error family. Regex is faster, has zero false-negative risk on exact patterns, and gives the ML model fewer ambiguous cases to handle.

> **Instructor Note:** This regex-first + ML-fallback pattern is common in production NLP systems. Think of it as a decision tree: if the log line contains a pattern we are 100% certain about, skip the model. Otherwise use the model. Combine both signals for highest precision.

In [8]:
REGEX_RULES = [
    # (compiled pattern, error_family, confidence)
    (re.compile(r'oom.kill|oom_kill|out.of.memory|balloon_inflation|heap_dump', re.I), 'MEMORY_ERROR',      0.95),
    (re.compile(r'ssl_handshake|cert_expired|rbac_deny|ldap.*bind_failed|lockout', re.I), 'AUTH_ERROR',    0.95),
    (re.compile(r'cerebro.*lag|cerebro.*fail|witness.*heartbeat|nearsync|snapshot.*skip', re.I), 'REPLICATION_ERROR', 0.95),
    (re.compile(r'disk.*io_error|io_error.*disk|wal.corrupt|bad_sector|marking_disk_bad', re.I), 'IO_ERROR', 0.95),
    (re.compile(r'nic_flap|lacp.*timeout|mtu_mismatch|ovs.*duplicate|link_state.*down', re.I), 'NETWORK_ERROR', 0.95),
    (re.compile(r'psu.*fail|fan.*rpm|ecc_error|chassis_intrusion|smart.*fail', re.I), 'HARDWARE_ERROR',     0.95),
]

def regex_classify(log_line: str) -> tuple:
    """Returns (family, confidence) if a regex rule matches, else (None, 0.0)."""
    for pattern, family, confidence in REGEX_RULES:
        if pattern.search(log_line):
            return family, confidence
    return None, 0.0

# Demonstrate
demo_logs = [
    "FATAL kernel OOM_killer invoked process=stargate rss=62GB available_mem=512MB",
    "ERROR cerebro replication lag_seconds=14400 threshold=3600 RPO_breach",
    "WARN ovs flow_table duplicate_entry src_mac vlan dropping_traffic",
    "ERROR prism api_gateway user rbac_deny policy_violation",
    "Unknown log line that doesn't match any pattern",
]
print(f'{"Log (truncated)":<55} {"Regex match":<18} Conf')
print('-' * 80)
for log in demo_logs:
    family, conf = regex_classify(log)
    match_str = family if family else 'no match → ML'
    print(f'{log[:53]:<55} {match_str:<18} {conf}')

Log (truncated)                                         Regex match        Conf
--------------------------------------------------------------------------------
FATAL kernel OOM_killer invoked process=stargate rss=   MEMORY_ERROR       0.95
ERROR cerebro replication lag_seconds=14400 threshold   REPLICATION_ERROR  0.95
WARN ovs flow_table duplicate_entry src_mac vlan drop   NETWORK_ERROR      0.95
ERROR prism api_gateway user rbac_deny policy_violati   AUTH_ERROR         0.95
Unknown log line that doesn't match any pattern         no match → ML      0.0


### B.3 Train the Log Classifier (ML Fallback)

In [9]:
# For log lines, we use the raw text (not lemmatised) because
# service names and error codes are already normalised tokens.

X_log = log_df['text'].values
y_log = log_df['family'].values

X_log_train, X_log_test, y_log_train, y_log_test = train_test_split(
    X_log, y_log, test_size=0.25, random_state=42, stratify=y_log
)

log_vec = TfidfVectorizer(max_features=300, ngram_range=(1, 2), sublinear_tf=True)
log_clf = CalibratedClassifierCV(LinearSVC(C=2.0, max_iter=2000, random_state=42), cv=3)

log_vec.fit(X_log_train)
log_clf.fit(log_vec.transform(X_log_train), y_log_train)

log_preds = log_clf.predict(log_vec.transform(X_log_test))
log_acc   = accuracy_score(y_log_test, log_preds)

print(f'Log classifier accuracy: {log_acc:.2%}')
print(classification_report(y_log_test, log_preds))

Log classifier accuracy: 27.27%
                   precision    recall  f1-score   support

       AUTH_ERROR       0.00      0.00      0.00         2
   HARDWARE_ERROR       0.00      0.00      0.00         1
         IO_ERROR       1.00      0.50      0.67         2
     MEMORY_ERROR       0.00      0.00      0.00         2
    NETWORK_ERROR       0.00      0.00      0.00         2
REPLICATION_ERROR       0.67      1.00      0.80         2

         accuracy                           0.27        11
        macro avg       0.28      0.25      0.24        11
     weighted avg       0.30      0.27      0.27        11



### B.4 Log Classification Function (Regex + ML combined)

In [10]:
LOG_ONCALL = {
    'IO_ERROR':          'storage-oncall@nutanix.internal',
    'REPLICATION_ERROR': 'dr-oncall@nutanix.internal',
    'NETWORK_ERROR':     'network-oncall@nutanix.internal',
    'MEMORY_ERROR':      'platform-oncall@nutanix.internal',
    'HARDWARE_ERROR':    'hardware-oncall@nutanix.internal',
    'AUTH_ERROR':        'security-oncall@nutanix.internal',
}

def classify_log(log_line: str) -> dict:
    """Classify an AOS log line using regex first, ML fallback."""
    # Regex pre-classifier
    family, conf = regex_classify(log_line)
    method = 'regex'

    if family is None:
        # ML fallback
        vec    = log_vec.transform([log_line])
        family = log_clf.predict(vec)[0]
        conf   = float(log_clf.predict_proba(vec)[0].max())
        method = 'ml'

    return {
        'family':    family,
        'oncall':    LOG_ONCALL.get(family, 'ops-oncall@nutanix.internal'),
        'confidence': round(conf, 3),
        'method':    method,
    }


# Test on realistic AOS log samples
test_logs = [
    "FATAL stargate disk_manager.cc:412] disk_id=sda3 io_error=EIO retry_count=3 marking_disk_bad",
    "FATAL kernel OOM_killer invoked process=stargate rss=62GB available_mem=512MB oom_score=900",
    "ERROR cerebro replication_manager.cc:203] site=DR-east pd=pd-prod-01 lag_seconds=14400",
    "ERROR ovs-vswitchd node_id=2 port=eth2 link_state=DOWN nic_flap_count=47",
    "FATAL ipmi sensor node_id=4 sensor=FAN_1 reading=0RPM hardware_failure_detected",
    "FATAL certificate ssl_handshake_failed prism_central cert_expired days_overdue=14",
    "WARN genesis timeout elapsed waiting for zookeeper quorum reconnecting",  # ambiguous
]

print(f'{"Log sample (truncated)":<55} {"Family":<20} {"Method":<7} Conf')
print('-' * 95)
for log in test_logs:
    result = classify_log(log)
    short  = log[:53] + '..' if len(log) > 53 else log
    print(f'{short:<55} {result["family"]:<20} {result["method"]:<7} {result["confidence"]}')

Log sample (truncated)                                  Family               Method  Conf
-----------------------------------------------------------------------------------------------
FATAL stargate disk_manager.cc:412] disk_id=sda3 io_e.. IO_ERROR             regex   0.95
FATAL kernel OOM_killer invoked process=stargate rss=.. MEMORY_ERROR         regex   0.95
ERROR cerebro replication_manager.cc:203] site=DR-eas.. REPLICATION_ERROR    regex   0.95
ERROR ovs-vswitchd node_id=2 port=eth2 link_state=DOW.. NETWORK_ERROR        regex   0.95
FATAL ipmi sensor node_id=4 sensor=FAN_1 reading=0RPM.. HARDWARE_ERROR       regex   0.95
FATAL certificate ssl_handshake_failed prism_central .. AUTH_ERROR           regex   0.95
WARN genesis timeout elapsed waiting for zookeeper qu.. IO_ERROR             ml      0.267


## Combined Pipeline — NutanixNLPPipeline

We wrap both classifiers into a single class with a clean interface. This is what would be deployed as a microservice or called from a Prism webhook.

> **Instructor Note:** The class encapsulates all state — both vectorisers, both classifiers, the routing table, and the regex rules. Serialising this single object with joblib gives you a complete, self-contained artefact. The REST API (Lab 3.2) would load this joblib file and expose `classify()` as an endpoint.

In [11]:
class NutanixNLPPipeline:
    """End-to-end NLP pipeline for Nutanix ticket routing and log classification."""

    INPUT_TICKET = 'ticket'
    INPUT_LOG    = 'log'

    def __init__(self, ticket_vec, ticket_clf, log_vec, log_clf,
                 routing_table, log_oncall, regex_rules, preprocessor):
        self.ticket_vec    = ticket_vec
        self.ticket_clf    = ticket_clf
        self.log_vec       = log_vec
        self.log_clf       = log_clf
        self.routing_table = routing_table
        self.log_oncall    = log_oncall
        self.regex_rules   = regex_rules
        self.preprocessor  = preprocessor

    def classify(self, text: str, input_type: str = None) -> dict:
        """Classify text as a ticket or log. Auto-detects type if input_type is None."""
        if input_type is None:
            input_type = self._detect_type(text)

        if input_type == self.INPUT_TICKET:
            return self._classify_ticket(text)
        else:
            return self._classify_log(text)

    def _detect_type(self, text: str) -> str:
        """Heuristic: AOS logs start with FATAL/ERROR/WARN followed by a service name."""
        if re.match(r'^(FATAL|ERROR|WARN|INFO)\s+\w', text.strip()):
            return self.INPUT_LOG
        return self.INPUT_TICKET

    def _classify_ticket(self, text: str) -> dict:
        clean     = self.preprocessor(text)
        vec       = self.ticket_vec.transform([clean])
        category  = self.ticket_clf.predict(vec)[0]
        conf      = float(self.ticket_clf.predict_proba(vec)[0].max())
        routing   = self.routing_table.get(category, {})
        return {
            'input_type': 'ticket',
            'category':   category,
            'queue':      routing.get('queue', 'L1-TRIAGE'),
            'team':       routing.get('team', 'L1 Manual Triage'),
            'sla_hours':  routing.get('sla_hours', 8),
            'confidence': round(conf, 3),
        }

    def _classify_log(self, text: str) -> dict:
        # Regex first
        for pattern, family, confidence in self.regex_rules:
            if pattern.search(text):
                return {
                    'input_type': 'log',
                    'family':     family,
                    'oncall':     self.log_oncall.get(family, 'ops-oncall@nutanix.internal'),
                    'confidence': confidence,
                    'method':     'regex',
                }
        # ML fallback
        vec    = self.log_vec.transform([text])
        family = self.log_clf.predict(vec)[0]
        conf   = float(self.log_clf.predict_proba(vec)[0].max())
        return {
            'input_type': 'log',
            'family':     family,
            'oncall':     self.log_oncall.get(family, 'ops-oncall@nutanix.internal'),
            'confidence': round(conf, 3),
            'method':     'ml',
        }


# Assemble the pipeline
pipeline = NutanixNLPPipeline(
    ticket_vec    = ticket_vec,
    ticket_clf    = ticket_clf,
    log_vec       = log_vec,
    log_clf       = log_clf,
    routing_table = ROUTING_TABLE,
    log_oncall    = LOG_ONCALL,
    regex_rules   = REGEX_RULES,
    preprocessor  = preprocess,
)

print('NutanixNLPPipeline assembled ✅')

NutanixNLPPipeline assembled ✅


## End-to-End Inference — Unified classify() Interface

> **Instructor Note:** Show the class receiving a mixed batch — some are support tickets, some are raw log lines — and auto-detecting the type. This is the real production scenario: a Prism webhook fires on any system event and the pipeline decides how to handle it.

In [12]:
mixed_inputs = [
    # Tickets (natural language)
    "Stargate has crashed on node-2, IOPS dropped to zero for all VMs",
    "Prism Central upgrade failed at the post-upgrade health check step",
    "Cerebro replication lag is 8 hours, RPO breach is imminent",
    "AHV live migration has been running for 45 minutes with no progress",
    # AOS log lines (structured prefix)
    "FATAL stargate disk_manager disk_id=sda3 io_error=EIO retry_count=3 marking_disk_bad",
    "ERROR ovs-vswitchd port=eth2 link_state=DOWN nic_flap_count=47 carrier_lost",
    "FATAL cerebro witness_heartbeat last_seen_secs=120 metro_cluster_at_risk",
    "FATAL certificate ssl_handshake_failed prism_central cert_expired days_overdue=14",
]

print('=== NutanixNLPPipeline — Unified Output ===')
for text in mixed_inputs:
    result = pipeline.classify(text)   # auto-detects input type
    short  = text[:60] + '..' if len(text) > 60 else text
    if result['input_type'] == 'ticket':
        print(f'\n[TICKET] {short}')
        print(f'  category={result["category"]}  queue={result["queue"]}  '
              f'team={result["team"]}  conf={result["confidence"]}')
    else:
        print(f'\n[LOG]    {short}')
        print(f'  family={result["family"]}  oncall={result["oncall"]}  '
              f'method={result["method"]}  conf={result["confidence"]}')

=== NutanixNLPPipeline — Unified Output ===

[TICKET] Stargate has crashed on node-2, IOPS dropped to zero for all..
  category=storage  queue=STOR-L2  team=Storage Engineering  conf=0.476

[TICKET] Prism Central upgrade failed at the post-upgrade health chec..
  category=prism  queue=PRISM-L1  team=Prism UI & API Support  conf=0.757

[TICKET] Cerebro replication lag is 8 hours, RPO breach is imminent
  category=replication  queue=DR-L2  team=Data Protection & DR  conf=0.696

[TICKET] AHV live migration has been running for 45 minutes with no p..
  category=compute  queue=CMP-L2  team=Compute & Hypervisor  conf=0.406

[LOG]    FATAL stargate disk_manager disk_id=sda3 io_error=EIO retry_..
  family=IO_ERROR  oncall=storage-oncall@nutanix.internal  method=regex  conf=0.95

[LOG]    ERROR ovs-vswitchd port=eth2 link_state=DOWN nic_flap_count=..
  family=NETWORK_ERROR  oncall=network-oncall@nutanix.internal  method=regex  conf=0.95

[LOG]    FATAL cerebro witness_heartbeat last_seen_secs=1

## Export the Pipeline as a Joblib Artefact

> **Instructor Note:** We save the entire `NutanixNLPPipeline` object — both classifiers, both vectorisers, and all supporting data — as a single joblib file. A FastAPI service (Lab 3.2 pattern) would load this at startup and call `pipeline.classify()` on each request. The pipeline card mirrors the model card from Lab 3.1.

In [13]:
import joblib, json

PIPELINE_PATH = 'nutanix_nlp_pipeline.pkl'
joblib.dump(pipeline, PIPELINE_PATH, compress=3)
size_kb = os.path.getsize(PIPELINE_PATH) / 1024
print(f'✅ Pipeline saved → {PIPELINE_PATH}  ({size_kb:.1f} KB)')

# Verify round-trip
loaded = joblib.load(PIPELINE_PATH)
test_result = loaded.classify("Stargate crashed after IO errors on node-2")
print(f'\nRound-trip check → {test_result}')

# Pipeline card
card = {
    'name': 'NutanixNLPPipeline',
    'version': '1.0.0',
    'ticket_categories': list(ROUTING_TABLE.keys()),
    'log_error_families': list(LOG_ONCALL.keys()),
    'ticket_vectoriser': 'TF-IDF (max_features=200, ngram_range=(1,2), sublinear_tf=True)',
    'log_vectoriser':    'TF-IDF (max_features=300, ngram_range=(1,2), sublinear_tf=True)',
    'ticket_classifier': 'LinearSVC (C=1.0) + CalibratedClassifierCV',
    'log_classifier':    'LinearSVC (C=2.0) + CalibratedClassifierCV',
    'log_regex_rules':   len(REGEX_RULES),
    'ticket_train_size': len(X_train),
    'log_train_size':    len(X_log_train),
    'ticket_accuracy':   f'{acc:.2%}',
    'log_accuracy':      f'{log_acc:.2%}',
    'artefact_kb':       round(size_kb, 1),
}
with open('nlp_pipeline_card.json', 'w') as f:
    json.dump(card, f, indent=2)

print('\n=== Pipeline Card ===')
print(json.dumps(card, indent=2))

✅ Pipeline saved → nutanix_nlp_pipeline.pkl  (30.4 KB)

Round-trip check → {'input_type': 'ticket', 'category': 'storage', 'queue': 'STOR-L2', 'team': 'Storage Engineering', 'sla_hours': 4, 'confidence': 0.678}

=== Pipeline Card ===
{
  "name": "NutanixNLPPipeline",
  "version": "1.0.0",
  "ticket_categories": [
    "storage",
    "network",
    "compute",
    "prism",
    "replication",
    "performance"
  ],
  "log_error_families": [
    "IO_ERROR",
    "REPLICATION_ERROR",
    "NETWORK_ERROR",
    "MEMORY_ERROR",
    "HARDWARE_ERROR",
    "AUTH_ERROR"
  ],
  "ticket_vectoriser": "TF-IDF (max_features=200, ngram_range=(1,2), sublinear_tf=True)",
  "log_vectoriser": "TF-IDF (max_features=300, ngram_range=(1,2), sublinear_tf=True)",
  "ticket_classifier": "LinearSVC (C=1.0) + CalibratedClassifierCV",
  "log_classifier": "LinearSVC (C=2.0) + CalibratedClassifierCV",
  "log_regex_rules": 6,
  "ticket_train_size": 45,
  "log_train_size": 31,
  "ticket_accuracy": "33.33%",
  "log_accuracy

## Lab Summary

### Module 4 — Applied NLP: Complete Picture

| Lab | Topic | Key output |
|-----|-------|------------|
| 4.1 | Text Preprocessing | `tickets_preprocessed.csv`, reusable `preprocess()` function |
| 4.2 | Text Representation & Classification | BoW vs TF-IDF comparison, NB / LR / SVM benchmarks |
| 4.3 | Production NLP Pipeline | `NutanixNLPPipeline` with ticket routing + log classification |

### Architecture recap

```
Incoming text
     │
     ├─ detect_type()  ─────────────────────────────┐
     │                                               │
  [ticket]                                        [log]
     │                                               │
  preprocess()                               regex_classify()
     │                                          ↓ no match
  TF-IDF vectorise                         TF-IDF vectorise
     │                                               │
  LinearSVC predict                         LinearSVC predict
     │                                               │
  routing_table lookup                     log_oncall lookup
     │                                               │
  {queue, team, sla}                    {family, oncall, method}
```

**Files produced:**
- `nutanix_nlp_pipeline.pkl` — complete serialised pipeline (joblib)
- `nlp_pipeline_card.json` — artefact metadata

> **Instructor Note:** Module 5 (GenAI APIs & RAG) will take this classifier one step further — instead of routing a ticket to a team, we'll use a RAT (retrieval-augmented) LLM to *draft the resolution* for the engineer. The NLP category from this lab becomes the retrieval key for the vector search.

---
## 🎯 Challenges

### Challenge 1 — Confidence Threshold Tuning
The `route_ticket()` function falls back to manual triage when confidence < 0.4. Plot a precision-recall curve for the ticket classifier and find the optimal threshold that maximises precision while routing at least 80% of tickets automatically.

### Challenge 2 — Regex Coverage Analysis
For the log dataset, calculate what percentage of log lines are handled by regex vs. ML. Add two additional regex rules that cover common patterns you observe in the ML-handled log lines. Show how this changes the regex/ML ratio.

### Challenge 3 — Priority Scoring
Add a `priority_score` field to the routing output for tickets. Use the SVM calibrated probability to compute:
- `priority = 'P1'` if confidence > 0.8 and category in `['network', 'replication']` (SLA = 2h)
- `priority = 'P2'` if confidence > 0.6
- `priority = 'P3'` otherwise

Update `NutanixNLPPipeline._classify_ticket()` to include `priority` in the output dict.

In [14]:
# Challenge workspace
